In [ ]:
%pip install yfinance

In [ ]:
dbutils.library.restartPython()

In [ ]:
import yfinance as yf
from databricks.sdk.runtime import dbutils

dbutils.widgets.text("ticker", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("catalog", "")

ticker = dbutils.widgets.get("ticker")
start_date = dbutils.widgets.get("start_date")
catalog = dbutils.widgets.get("catalog")

In [ ]:
try:
    if start_date:
        prices = yf.download(ticker, start=start_date, interval="1d", auto_adjust=False)
    else:
        prices = yf.download(ticker, period="max", interval="1d", auto_adjust=False)

    prices = prices.reset_index()
    prices.columns = [c.lower().replace(" ", "_") for c in prices.columns]

    table = ticker.replace(".", "_")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.yfinance")
    spark.createDataFrame(prices).write.format("delta").mode("append").saveAsTable(
        f"{catalog}.yfinance.{table}"
    )

    dbutils.notebook.exit("0")
except Exception as e:
    print(f"{ticker} ingestion failed: {e}")
    dbutils.notebook.exit("1")